In [133]:
import pandas as pd

# Define the data
data = {
    'customer_id': [1, 1, 2, 2, 3],
    'order_amount': [100, 200, 150, 50, 300],
    'order_date': ['2024-01-01', '2024-01-05', '2024-01-03', '2024-01-10', '2024-01-02']
}

# Create the DataFrame
df = pd.DataFrame(data)

# Convert 'order_date' to actual datetime objects
df['order_date'] = pd.to_datetime(df['order_date'])

print(df)

   customer_id  order_amount order_date
0            1           100 2024-01-01
1            1           200 2024-01-05
2            2           150 2024-01-03
3            2            50 2024-01-10
4            3           300 2024-01-02


Task 1 (Basic Filtering + Aggregation):

1. Find total order amount per customer

2. Return only customers with total > 200

In [134]:
df['total_order_amount'] = df.groupby('customer_id')['order_amount'].transform('sum')
df

,customer_id,order_amount,order_date,total_order_amount
0,1,100,2024-01-01,300
1,1,200,2024-01-05,300
2,2,150,2024-01-03,200
3,2,50,2024-01-10,200
4,3,300,2024-01-02,300


In [135]:
df.loc[df['total_order_amount'] > 200]

,customer_id,order_amount,order_date,total_order_amount
0,1,100,2024-01-01,300
1,1,200,2024-01-05,300
4,3,300,2024-01-02,300


Corrected code suggested in review.

- Transform returns row level output of the initial dataframe. It is not needed for the answer
- Think about how the stakeholders could easily read the output

In [136]:
result = (
    df.groupby('customer_id', as_index=False)['order_amount']
      .sum()
      .rename(columns={'order_amount': 'total_order_amount'})
)

result = result[result['total_order_amount'] > 200]
result

,customer_id,total_order_amount
0,1,300
2,3,300


Task 2 (Groupby + Ranking):

1. For each customer, find their largest order

2. Then return the top 2 customers based on that largest order

In [137]:
df.groupby('customer_id')['order_amount'].max()

customer_id
1    200
2    150
3    300
Name: order_amount, dtype: int64

In [138]:
df.groupby('customer_id')['order_amount'].max().sort_values(ascending = False).head(2)

customer_id
3    300
1    200
Name: order_amount, dtype: int64

In [139]:
df.groupby('customer_id', as_index=False)['order_amount'].max().rename(columns={'order_amount': 'largest_order'})

,customer_id,largest_order
0,1,200
1,2,150
2,3,300


Cleaner answer suggested in review.
- Returns a dataframe, 
- renamed columns, 
- production friendly

In [140]:
result = (
    df.groupby('customer_id', as_index=False)['order_amount']
      .max()
      .rename(columns={'order_amount': 'largest_order'})
      .sort_values(by='largest_order', ascending=False)
      .head(2)
)
result

,customer_id,largest_order
2,3,300
0,1,200


In [141]:
# Define the data in a dictionary
data = {
    'customer_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie']
}

# Create the DataFrame
customers = pd.DataFrame(data)

# Display the table
print(customers)

# Define the orders data
orders_data = {
    'order_id': [101, 102, 103, 104],
    'customer_id': [1, 1, 2, 4],
    'amount': [100, 200, 50, 300]
}

# Create the DataFrame
orders = pd.DataFrame(orders_data)

print(orders)



   customer_id     name
0            1    Alice
1            2      Bob
2            3  Charlie
   order_id  customer_id  amount
0       101            1     100
1       102            1     200
2       103            2      50
3       104            4     300


Task 3 (Merge + Filtering):

- Join customers with orders (keep only valid matches)
- Calculate total order amount per customer
    
    Return:

        - customer_id

        - name

        - total_amount

- Only include customers with total_amount ≥ 150

In [142]:
merged = customers.merge(orders, on = 'customer_id', how = 'inner')
merged

,customer_id,name,order_id,amount
0,1,Alice,101,100
1,1,Alice,102,200
2,2,Bob,103,50


In [149]:
result = merged.groupby(['customer_id', 'name'], as_index=False)['amount'].sum()
result

,customer_id,name,amount
0,1,Alice,300
1,2,Bob,50


In [144]:
result = result.loc[result['amount'] >= 150 ]
result

,customer_id,amount
0,1,300


Task 4 (transform + ranking):

- For each customer, add a column:

       - running_total → cumulative sum of orders by date

- For each customer, rank their orders:

       - Highest order = rank 1

       - Use dense ranking (no gaps)

- Return only the top order per customer using your ranking



In [145]:
# Create the DataFrame
df = pd.DataFrame({
    'customer_id': [1, 1, 1, 2, 2],
    'order_amount': [100, 200, 50, 300, 100],
    'order_date': ['2024-01-01', '2024-01-05', '2024-01-07', '2024-01-02', '2024-01-06']
})

# Convert order_date to datetime objects for better analysis
df['order_date'] = pd.to_datetime(df['order_date'])

print(df)


   customer_id  order_amount order_date
0            1           100 2024-01-01
1            1           200 2024-01-05
2            1            50 2024-01-07
3            2           300 2024-01-02
4            2           100 2024-01-06


In [146]:
df = df.set_index('order_date')
df['cumulative_sum'] = df.groupby('customer_id')['order_amount'].cumsum()
df

,customer_id,order_amount,cumulative_sum
order_date,,,
2024-01-01,1,100,100
2024-01-05,1,200,300
2024-01-07,1,50,350
2024-01-02,2,300,300
2024-01-06,2,100,400


In [147]:
#df = df.drop('cumulative_sum', axis = 1)
df = df.reset_index('order_date')
df['rank_per_customer'] = (df.groupby('customer_id')['order_amount']
                           .rank(method='dense', ascending = False))
df

,order_date,customer_id,order_amount,cumulative_sum,rank_per_customer
0,2024-01-01,1,100,100,2.0
1,2024-01-05,1,200,300,1.0
2,2024-01-07,1,50,350,3.0
3,2024-01-02,2,300,300,1.0
4,2024-01-06,2,100,400,2.0


In [148]:
df = df.loc[df['rank_per_customer'] == 1]
df

,order_date,customer_id,order_amount,cumulative_sum,rank_per_customer
1,2024-01-05,1,200,300,1.0
3,2024-01-02,2,300,300,1.0


Cleaner answer suggested in review.

- setting date index is not necessary in this scenario
- sort data explicitly

In [150]:
# Step 1: Sort for correct running total
df = df.sort_values(['customer_id', 'order_date'])

# Step 2: Running total
df['running_total'] = df.groupby('customer_id')['order_amount'].cumsum()

# Step 3: Dense rank
df['rank'] = (
    df.groupby('customer_id')['order_amount']
      .rank(method='dense', ascending=False)
)

# Step 4: Top order per customer
top_orders = df[df['rank'] == 1]